# Data Exploration Part 2: Descriptive Statistics Analysis
## Kairos Project - Fadel Transportes Fleet Maintenance Prediction

**Team:** Ilariê  
**Objective:** Comprehensive descriptive statistics for all datasets  
**Focus:** Mean, median, standard deviation, min, max, null counts, and frequency analysis  
**Note:** Fixed version with robust error handling


## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

In [ ]:
# Load all datasets
try:
    df_service = pd.read_excel('data/SERVICE_ORDER_BASE.xlsx')
    df_vehicles = pd.read_excel('data/VEHICLES_BASE.xlsx')

    datasets = {
        'Service Orders': df_service,
        'Vehicle Master Data': df_vehicles
    }

    print("=== DATASETS OVERVIEW ===")
    for name, df in datasets.items():
        print(f"{name}: {df.shape[0]:,} records × {df.shape[1]} columns")

    total_records = sum(df.shape[0] for df in datasets.values())
    total_columns = sum(df.shape[1] for df in datasets.values())
    print(f"\nTotal Combined: {total_records:,} records × {total_columns} columns")

except Exception as e:
    print(f"Error loading datasets: {e}")
    print("Please ensure the data files are in the correct location.")

## 2. Comprehensive Descriptive Statistics Function

### 3.1. Service Orders Insights

#### **Motivação:**
A análise dos service orders é fundamental para entender os padrões de manutenção da frota da Fadel Transportes. Esta seção visa identificar:

- **Distribuição de custos**: Compreender a variabilidade dos custos de manutenção
- **Padrões categóricos**: Identificar as principais características dos service orders
- **Qualidade dos dados**: Avaliar a completude das informações financeiras
- **Insights operacionais**: Extrair informações relevantes para tomada de decisão

Esta análise é essencial para fundamentar estratégias de manutenção preventiva e otimização de custos operacionais.


In [ ]:
def safe_describe_dataset(df, dataset_name):
    """
    Safely generate comprehensive descriptive statistics for a dataset
    """
    print(f"=== {dataset_name.upper()} DESCRIPTIVE STATISTICS ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)[:5]}..." if len(df.columns) > 5 else f"Columns: {list(df.columns)}")

    # Separate numerical and categorical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(include=['object', 'datetime64[ns]']).columns

    print(f"\nNumerical columns: {len(numerical_cols)}")
    print(f"Categorical columns: {len(categorical_cols)}")

    # Create comprehensive statistics table
    stats_summary = []

    for col in df.columns:
        try:
            col_stats = {
                'Column': col,
                'Data_Type': str(df[col].dtype),
                'Non_Null_Count': df[col].count(),
                'Null_Count': df[col].isnull().sum(),
                'Null_Percentage': (df[col].isnull().sum() / len(df)) * 100,
                'Unique_Values': df[col].nunique()
            }

            # Add numerical statistics if column is numeric
            if pd.api.types.is_numeric_dtype(df[col]) and df[col].count() > 0:
                col_stats.update({
                    'Mean': df[col].mean(),
                    'Median': df[col].median(),
                    'Std_Dev': df[col].std(),
                    'Min': df[col].min(),
                    'Max': df[col].max(),
                    'Q1': df[col].quantile(0.25),
                    'Q3': df[col].quantile(0.75)
                })
            else:
                # Add categorical statistics
                if df[col].count() > 0:
                    mode_value = df[col].mode().iloc[0] if len(df[col].mode()) > 0 else None
                    most_frequent_count = df[col].value_counts().iloc[0] if len(df[col].value_counts()) > 0 else 0
                    col_stats.update({
                        'Most_Frequent_Value': mode_value,
                        'Most_Frequent_Count': most_frequent_count,
                        'Most_Frequent_Percentage': (most_frequent_count / len(df)) * 100
                    })

            stats_summary.append(col_stats)

        except Exception as e:
            print(f"Error processing column {col}: {e}")
            continue

    return pd.DataFrame(stats_summary)

In [ ]:
# Cost Distribution Analysis - Coefficient of Variation Calculation
print("\n=== COST DISTRIBUTION ANALYSIS ===")
if 'GRAND TOTAL' in df_service.columns:
    cost_data = df_service['GRAND TOTAL'].dropna()
    if len(cost_data) > 0:
        mean_cost = cost_data.mean()
        std_cost = cost_data.std()
        median_cost = cost_data.median()
        coefficient_of_variation = std_cost / mean_cost
        
        print(f"Cost Statistics:")
        print(f"   • Mean: R$ {mean_cost:.2f}")
        print(f"   • Median: R$ {median_cost:.2f}")
        print(f"   • Standard Deviation: R$ {std_cost:.2f}")
        print(f"   • Coefficient of Variation: {coefficient_of_variation:.2f}")
        print(f"   • Min: R$ {cost_data.min():.2f}")
        print(f"   • Max: R$ {cost_data.max():.2f}")
        
        print(f"\nInterpretation:")
        print(f"   • CV = {coefficient_of_variation:.2f} indicates {'high' if coefficient_of_variation > 1 else 'moderate' if coefficient_of_variation > 0.5 else 'low'} variability")
        print(f"   • Right-skewed distribution: median (R$ {median_cost:.2f}) < mean (R$ {mean_cost:.2f})")
        print(f"   • Most maintenance events are relatively inexpensive, with few high-cost interventions")


In [ ]:
# Fleet Composition Insights Analysis
print("\n=== FLEET COMPOSITION INSIGHTS ===")

# Vehicle Models Analysis
if 'DESCRICAO MODELO' in df_service.columns:
    model_counts = df_service['DESCRICAO MODELO'].value_counts()
    total_models = len(model_counts)
    top_model = model_counts.index[0]
    top_model_count = model_counts.iloc[0]
    top_model_percentage = (top_model_count / len(df_service)) * 100
    
    print(f"Vehicle Models Analysis:")
    print(f"   • Total unique models: {total_models}")
    print(f"   • Most frequent model: {top_model}")
    print(f"   • Top model frequency: {top_model_count:,} ({top_model_percentage:.1f}%)")
    print(f"   • Top 5 models represent: {(model_counts.head(5).sum() / len(df_service) * 100):.1f}% of all service orders")

# Part Categories Analysis
if 'DESCRICAO PRODUTO' in df_service.columns:
    product_counts = df_service['DESCRICAO PRODUTO'].value_counts()
    total_products = len(product_counts)
    top_product = product_counts.index[0]
    top_product_count = product_counts.iloc[0]
    top_product_percentage = (top_product_count / len(df_service)) * 100
    
    print(f"\nPart Categories Analysis:")
    print(f"   • Total unique products: {total_products}")
    print(f"   • Most frequent product: {top_product}")
    print(f"   • Top product frequency: {top_product_count:,} ({top_product_percentage:.1f}%)")
    print(f"   • Top 10 products represent: {(product_counts.head(10).sum() / len(df_service) * 100):.1f}% of all service orders")

# Brand Strategy Analysis
if 'MANUFACTURER NAME' in df_service.columns:
    manufacturer_counts = df_service['MANUFACTURER NAME'].value_counts()
    total_manufacturers = len(manufacturer_counts)
    top_manufacturer = manufacturer_counts.index[0]
    top_manufacturer_count = manufacturer_counts.iloc[0]
    top_manufacturer_percentage = (top_manufacturer_count / len(df_service)) * 100
    
    print(f"\nBrand Strategy Analysis:")
    print(f"   • Total manufacturers: {total_manufacturers}")
    print(f"   • Top manufacturer: {top_manufacturer}")
    print(f"   • Top manufacturer share: {top_manufacturer_count:,} ({top_manufacturer_percentage:.1f}%)")
    print(f"   • Top 3 manufacturers represent: {(manufacturer_counts.head(3).sum() / len(df_service) * 100):.1f}% of all service orders")
    
    # Calculate concentration index (Herfindahl-Hirschman Index)
    market_shares = manufacturer_counts / len(df_service)
    hhi = (market_shares ** 2).sum()
    print(f"   • Market concentration (HHI): {hhi:.4f} ({'High' if hhi > 0.25 else 'Moderate' if hhi > 0.15 else 'Low'} concentration)")

print(f"\nBusiness Implications:")
print(f"   • Model standardization enables economies of scale in parts procurement")
print(f"   • Product concentration suggests standardized maintenance procedures")
print(f"   • Brand concentration allows for specialized maintenance expertise")
print(f"   • High concentration indicates potential for strategic supplier relationships")


#### **Conclusões:**

Com base na análise dos service orders, identificamos os seguintes insights operacionais:

**Distribuição de Custos:**
- A variabilidade dos custos de manutenção indica diferentes tipos de intervenções
- A diferença entre média e mediana sugere a presença de eventos de alto custo
- A completude dos dados financeiros é fundamental para análises precisas

**Padrões Categóricos:**
- A concentração em determinadas marcas/manufacturers indica estratégias de padronização
- A proporção entre manutenção preventiva vs corretiva revela oportunidades de otimização
- O status dos ativos reflete a condição operacional da frota

**Implicações para o Negócio:**
- Focar recursos em marcas com maior representatividade pode gerar economias de escala
- Aumentar a proporção de manutenção preventiva pode reduzir custos operacionais
- Monitorar a qualidade dos dados financeiros é essencial para tomada de decisão

Estes insights fornecem a base para o desenvolvimento de estratégias de manutenção preditiva e otimização de custos.


## 3. Dataset 1: Service Orders

In [ ]:
# Generate statistics for service orders
service_stats = safe_describe_dataset(df_service, 'Service Orders Dataset')
display(service_stats)

# Focus on key metrics
print("\n=== SERVICE ORDERS KEY METRICS ===")
if 'GRAND TOTAL' in df_service.columns:
    non_null_costs = df_service['GRAND TOTAL'].dropna()
    if len(non_null_costs) > 0:
        print(f"Average service cost: R$ {non_null_costs.mean():.2f}")
        print(f"Median service cost: R$ {non_null_costs.median():.2f}")
        print(f"Max service cost: R$ {non_null_costs.max():.2f}")
        print(f"Missing cost data: {df_service['GRAND TOTAL'].isnull().sum():,} ({df_service['GRAND TOTAL'].isnull().sum()/len(df_service)*100:.1f}%)")

# Key categorical insights
print("\nSERVICE ORDERS INSIGHTS:")
key_cats = ['MANUFACTURER NAME', 'PREVENTIVE_CORRECTIVE MAINTENANCE', 'MAINTENANCE TYPE', 'ASSET STATUS']
for col in key_cats:
    if col in df_service.columns:
        top_value = df_service[col].value_counts().index[0]
        top_count = df_service[col].value_counts().iloc[0]
        percentage = (top_count / len(df_service)) * 100
        print(f"   • {col}: {top_value} ({top_count:,}, {percentage:.1f}%)")

## 4. Dataset 2: Vehicle Master Data

In [ ]:
# Generate statistics for vehicles
vehicles_stats = safe_describe_dataset(df_vehicles, 'Vehicle Master Dataset')
display(vehicles_stats)

# Vehicle age analysis with robust error handling
current_year = 2024
manufacture_year_col = None

# Find the manufacture year column (handle variations)
possible_cols = ['MANUFACTURE YEAR', 'MANUFACTURE_YEAR', 'ANO_FABRICACAO']
for col in possible_cols:
    if col in df_vehicles.columns:
        manufacture_year_col = col
        break

if manufacture_year_col and not df_vehicles[manufacture_year_col].isnull().all():
    # Calculate vehicle age safely
    valid_years = df_vehicles[manufacture_year_col].dropna()
    if len(valid_years) > 0:
        vehicle_ages = current_year - valid_years

        print(f"\nVEHICLE AGE ANALYSIS (based on {manufacture_year_col}):")
        print(f"   • Average age: {vehicle_ages.mean():.1f} years")
        print(f"   • Age range: {vehicle_ages.min():.0f} - {vehicle_ages.max():.0f} years")
        print(f"   • Median age: {vehicle_ages.median():.1f} years")

        # Age distribution
        age_0_5 = (vehicle_ages <= 5).sum()
        age_6_10 = ((vehicle_ages >= 6) & (vehicle_ages <= 10)).sum()
        age_11_15 = ((vehicle_ages >= 11) & (vehicle_ages <= 15)).sum()
        age_over_15 = (vehicle_ages > 15).sum()

        print(f"\n   Age Distribution:")
        print(f"     - 0-5 years: {age_0_5:,} vehicles ({age_0_5/len(valid_years)*100:.1f}%)")
        print(f"     - 6-10 years: {age_6_10:,} vehicles ({age_6_10/len(valid_years)*100:.1f}%)")
        print(f"     - 11-15 years: {age_11_15:,} vehicles ({age_11_15/len(valid_years)*100:.1f}%)")
        print(f"     - >15 years: {age_over_15:,} vehicles ({age_over_15/len(valid_years)*100:.1f}%)")
else:
    print(f"\nCannot calculate vehicle age - manufacture year column not found or empty")
    print(f"Available columns: {list(df_vehicles.columns)}")

# Fleet status analysis
if 'ASSET STATUS' in df_vehicles.columns:
    print(f"\nFLEET STATUS:")
    status_counts = df_vehicles['ASSET STATUS'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / len(df_vehicles)) * 100
        print(f"   • {status}: {count:,} vehicles ({percentage:.1f}%)")

## 5. Cross-Dataset Summary

In [ ]:
# Create comprehensive summary
all_stats = []
dataset_names = ['Service Orders', 'Vehicle Master']
dfs = [df_service, df_vehicles]

for name, df in zip(dataset_names, dfs):
    stats = safe_describe_dataset(df, name)
    all_stats.append(stats)

# Combine all statistics
if all_stats:
    combined_stats = pd.concat(all_stats, ignore_index=True)

    print("=== CROSS-DATASET SUMMARY ===")

    # Dataset overview
    summary_data = []
    for name, df in zip(dataset_names, dfs):
        numerical_cols = len(df.select_dtypes(include=[np.number]).columns)
        categorical_cols = len(df.select_dtypes(include=['object', 'datetime64[ns]']).columns)
        missing_cells = df.isnull().sum().sum()
        total_cells = df.shape[0] * df.shape[1]
        completeness = ((total_cells - missing_cells) / total_cells) * 100

        summary_data.append({
            'Dataset': name,
            'Records': df.shape[0],
            'Columns': df.shape[1],
            'Numerical': numerical_cols,
            'Categorical': categorical_cols,
            'Completeness_%': completeness
        })

    summary_df = pd.DataFrame(summary_data)
    display(summary_df)

    # Overall statistics
    total_records = sum(df.shape[0] for df in dfs)
    total_columns = sum(df.shape[1] for df in dfs)
    avg_completeness = summary_df['Completeness_%'].mean()

    print(f"\nOVERALL SUMMARY:")
    print(f"   • Total records analyzed: {total_records:,}")
    print(f"   • Total columns analyzed: {total_columns}")
    print(f"   • Average data completeness: {avg_completeness:.1f}%")

    # Find high missing data columns
    high_missing = combined_stats[combined_stats['Null_Percentage'] > 10]
    if len(high_missing) > 0:
        print(f"\nColumns with >10% missing data: {len(high_missing)}")
        for _, row in high_missing.iterrows():
            print(f"     {row['Column']}: {row['Null_Percentage']:.1f}% missing")

else:
    print("No statistics generated - check data loading.")